# Module 3 — Exploratory Data Analysis

**Modular Daraz Analysis — Module 3 of 6**

**Purpose.** Use pandas to compare categories, prices, sold counts, ratings and offers, then add the project price groups for downstream analysis.

**Course concepts visible in this part:** pandas `groupby()`, `agg()`, `value_counts()`, `nlargest()`, median/mean, filtering, lists, loops, conditions, imported reusable functions and file output.

**Project rule:** This notebook is one stage of the same Daraz analysis. It does not use separate practice examples; every concept is applied to the real project data.

## Load the cleaned output from Module 2
The analysis stage starts from the saved cleaned dataset. This keeps data preparation separate from exploratory analysis.

In [1]:
import pandas as pd
from daraz_utils import price_group

df = pd.read_csv("daraz_cleaned_data.csv")
print("Cleaned shape:", df.shape)
df.head()

Cleaned shape: (1051, 11)


,Title,Rating in Stars,Rating Count,Sold Count,Delivery,Current Price,Original Price,Category,Has Voucher,Discount,Discount %
0,Yfashion Cartoon Electronic Watch Life Water...,4.6,707,6000.0,Free Delivery,287,520.0,Kids Watches,No,233.0,44.807692
1,"100% Imported LED watch for men , boys and Kid...",4.3,293,1000.0,No Free Delivery,270,999.0,Kids Watches,No,729.0,72.972973
2,1 Cartoon Characters Analog Wrist Watch For Ki...,4.5,57,452.0,Free Delivery,225,750.0,Kids Watches,No,525.0,70.000000
3,M3 Touch LED Bracelet Digital Watch Band Good ...,4.6,51,332.0,Free Delivery,160,299.0,Kids Watches,No,139.0,46.488294
4,led Watches For Boys and Girls and kids,4.4,54,476.0,Free Delivery,198,300.0,Kids Watches,No,102.0,34.000000


## Pandas exploratory analysis
These cells answer the project's core descriptive questions with `value_counts()`, `groupby()`, aggregation, sorting and filtering.

In [2]:
# which category has the most products
df["Category"].value_counts()

Category
Womens Bags           199
Mens Watches          160
Mens Accessories      160
Womens Watches        160
Kids Watches          132
Mens Bags             120
Womens Accessories    120
Name: count, dtype: int64

In [3]:
# overall price
print("cheapest:", df["Current Price"].min())
print("most expensive:", df["Current Price"].max())
print("average:", round(df["Current Price"].mean(), 2))
print("middle:", df["Current Price"].median())

cheapest: 60
most expensive: 9999
average: 705.68
middle: 525.0


**Why median matters here.** The mean price is much higher than the median because a small number of expensive products stretch the distribution to the right. For a skewed marketplace dataset, the median gives a better picture of a typical listing than the mean.

In [4]:
# price for each category
df.groupby("Category")["Current Price"].median().sort_values(ascending=False)

Category
Womens Bags           899.0
Womens Watches        599.0
Mens Watches          555.0
Mens Bags             499.0
Mens Accessories      465.0
Womens Accessories    332.5
Kids Watches          302.5
Name: Current Price, dtype: float64

**How `groupby` works here.** It splits the cleaned product records into the 7 categories and calculates the median separately inside each group. That lets us compare a typical product in one category with a typical product in another category.

In [5]:
# how much sells in each category
df.groupby("Category")["Sold Count"].median().sort_values(ascending=False)

Category
Mens Bags             662.5
Womens Accessories    588.0
Womens Bags           497.5
Mens Accessories      454.0
Mens Watches          440.0
Womens Watches        399.0
Kids Watches          135.0
Name: Sold Count, dtype: float64

In [6]:
# rating for each category
df.groupby("Category")["Rating in Stars"].mean().round(2).sort_values(ascending=False)

Category
Womens Accessories    4.47
Womens Bags           4.47
Mens Bags             4.37
Kids Watches          4.36
Mens Accessories      4.35
Womens Watches        4.34
Mens Watches          4.32
Name: Rating in Stars, dtype: float64

In [7]:
# a compact category-level summary
summary = df.groupby("Category").agg(
    products=("Title", "count"),
    median_price=("Current Price", "median"),
    median_sold=("Sold Count", "median"),
    average_rating=("Rating in Stars", "mean"),
    median_rating_count=("Rating Count", "median"),
    median_discount_pct=("Discount %", "median")
).round(2)

summary.sort_values("median_price", ascending=False)

,products,median_price,median_sold,average_rating,median_rating_count,median_discount_pct
Category,,,,,,
Womens Bags,199,899.0,497.5,4.47,119.0,42.68
Womens Watches,160,599.0,399.0,4.34,59.5,44.89
Mens Watches,160,555.0,440.0,4.32,69.5,44.15
Mens Bags,120,499.0,662.5,4.37,122.0,50.15
Mens Accessories,160,465.0,454.0,4.35,74.0,47.67
Womens Accessories,120,332.5,588.0,4.47,80.5,50.00
Kids Watches,132,302.5,135.0,4.36,12.5,40.94


**What this table is for.** This combines the most useful category-level measures in one place: number of records, typical price, typical sold count, average star rating, typical rating volume and typical discount percentage. It makes category comparisons easier without replacing the more detailed relationship checks below.

In [8]:
# does free delivery change anything
df.groupby("Delivery")[["Current Price", "Sold Count", "Rating in Stars"]].median()

,Current Price,Sold Count,Rating in Stars
Delivery,,,
Free Delivery,580.0,446.5,4.4
No Free Delivery,428.5,441.0,4.5


In [9]:
# do vouchers change anything
df.groupby("Has Voucher")[["Current Price", "Sold Count", "Rating in Stars"]].median()

,Current Price,Sold Count,Rating in Stars
Has Voucher,,,
No,484.0,436.0,4.4
Yes,779.0,515.0,4.5


**Initial offer comparison.** At the overall level, listings with a voucher have a higher median sold count than listings without one, while the difference between free-delivery and no-free-delivery listings is very small.

These are **descriptive associations**, not causal effects. Categories have different mixes of vouchers, delivery offers, prices and popularity, so we also inspect the patterns by category later instead of concluding that the offer itself caused the difference.

In [10]:
# the 5 records with the highest reported sold count
df.nlargest(5, "Sold Count")[[
    "Title", "Category", "Sold Count", "Current Price", "Rating in Stars", "Rating Count"
]]

,Title,Category,Sold Count,Current Price,Rating in Stars,Rating Count
137,2 Piece Beanie Cap + Neck Warmer/Hat & Warmer ...,Mens Accessories,35000.0,459,4.6,5553
157,Winter Cap and Winter beanie cap for men,Mens Accessories,33000.0,476,4.6,5018
132,Beanie cap-Wool cap with neck warmer for men w...,Mens Accessories,18000.0,447,4.3,2258
292,Slim and light weight Long Wallet For Men,Mens Bags,18000.0,398,4.5,2720
453,SKMEI Sports Fashion Quartz Dual Display Water...,Mens Watches,17000.0,3699,4.7,4269


In [11]:
# useful dataset-level checks
print("sum of available reported sold counts:", int(df["Sold Count"].sum()))
print("products with no customer rating yet:", (df["Rating Count"] == 0).sum())

sum of available reported sold counts: 1117548
products with no customer rating yet: 11


## Applying a reusable function, loop and conditions
`price_group()` is imported from `daraz_utils.py`. The cut-offs are analyst-defined project categories, not official Daraz labels.

**How we picked the cut-off points.** Rs 500 is close to the overall median price, so it creates a useful lower-price group. Rs 2000 separates the small high-price tail visible in the price distribution. The cut-offs are simple analysis categories, not official Daraz market segments.

In [12]:
groups = []

for price in df["Current Price"]:
    groups.append(price_group(price))

df["Price Group"] = groups
df["Price Group"].value_counts()

Price Group
Budget       515
Mid          498
Expensive     38
Name: count, dtype: int64

In [13]:
# check the new column looks right
df[["Title", "Current Price", "Price Group"]].head()

,Title,Current Price,Price Group
0,Yfashion Cartoon Electronic Watch Life Water...,287,Budget
1,"100% Imported LED watch for men , boys and Kid...",270,Budget
2,1 Cartoon Characters Analog Wrist Watch For Ki...,225,Budget
3,M3 Touch LED Bracelet Digital Watch Band Good ...,160,Budget
4,led Watches For Boys and Girls and kids,198,Budget


We print a few rows to confirm the groups line up with the prices, rather than trusting that
the function worked.

In [14]:
# compare the three groups
df.groupby("Price Group")[["Sold Count", "Rating in Stars"]].median()

,Sold Count,Rating in Stars
Price Group,,
Budget,454.0,4.4
Expensive,745.0,4.7
Mid,399.0,4.5


In [15]:
# use a loop and an if/else condition to print selected category summaries
categories = ["Womens Bags", "Mens Watches", "Kids Watches"]
overall_median_price = df["Current Price"].median()

for category in categories:
    data = df[df["Category"] == category]
    price = data["Current Price"].median()
    sold = data["Sold Count"].median()

    if price > overall_median_price:
        note = "above overall median price"
    else:
        note = "at or below overall median price"

    print(category, "|", len(data), "products | median price", price,
          "| median sold", sold, "|", note)

Womens Bags | 199 products | median price 899.0 | median sold 497.5 | above overall median price
Mens Watches | 160 products | median price 555.0 | median sold 440.0 | above overall median price
Kids Watches | 132 products | median price 302.5 | median sold 135.0 | at or below overall median price


**What this loop does.** The list stores three categories, the `for` loop processes them one at a time, and the `if / else` condition compares each category's median price with the overall median price. This is the same Python logic taught in class, but used directly on the Daraz analysis rather than in a separate example.

## Save the analysis-ready dataset
Module 3 adds `Price Group`, which the visualization module uses later. Saving this stage creates a clean dependency between modules.

In [16]:
df.to_csv("daraz_analysis_ready.csv", index=False)
print("Analysis-ready dataset saved: daraz_analysis_ready.csv")

Analysis-ready dataset saved: daraz_analysis_ready.csv
